<a href="https://colab.research.google.com/github/yuviyadav-git/Miniprojects/blob/main/Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a Project on Vector Search, I mostly followed this tutorial ([Vector Search RAG Tutorial]https://www.youtube.com/watch?v=JEBDfGqrAUA&t=1807s), it has a few changes from my side, those changes are marked below.

Here is the link to the original code for reference : https://github.com/beaucarnes/vector-search-tutorial/tree/main/project-one

In [ ]:
!pip install pymongo
!pip install huggingface_hub
!pip install sentence_transformers
import os
import pymongo

In [ ]:
client=pymongo.MongoClient("mongodb+srv://<user_id>:<password>@cluster0.oq3vnum.mongodb.net/?appName=Cluster0")
db=client.sample_mflix
collection=db.movies

This is the very first change, in the original code a generate_embedding function was used, and requests was used to get the embedding from the hugging face url.

I did write that initially, but after studying a bit about sentence_transformers, i found it can be run locally, it is only 90mb, so i intalled sentence_transformers instead.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embedding = model.encode("I am yuvi")

print(embedding)
print("Vector Shape:", embedding.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-7.26781040e-02 -1.18447235e-02  4.63828184e-02  2.34107971e-02
 -3.11539490e-02 -8.32479298e-02  1.02692179e-01  3.87200434e-03
  6.18729182e-03 -1.33572603e-02 -5.59613816e-02 -9.75739509e-02
  1.20429704e-02  3.09916101e-02  7.90199563e-02 -8.36443678e-02
 -4.46698964e-02  3.20165977e-02 -4.26335298e-02 -3.56465243e-02
 -6.69532642e-02 -5.10582933e-03  2.40640473e-02 -2.36619953e-02
 -9.17121302e-03  1.39456254e-03  6.45956770e-02  4.91423979e-02
  6.87918290e-02 -9.96360928e-02 -1.62028428e-03  6.97139502e-02
  2.98909135e-02 -2.51437332e-02 -1.47822484e-01  5.40550351e-02
  5.13199195e-02 -5.45934252e-02 -1.20843388e-02  8.09806213e-03
 -1.32784694e-02  5.26182307e-03  3.15760747e-02 -2.11949600e-03
  1.39860362e-01 -6.43600523e-02 -5.80464453e-02 -1.38798691e-02
  6.68332539e-03  1.68576986e-02 -2.72516515e-02 -2.32118368e-02
 -4.42575710e-03  4.35153283e-02  5.58437817e-02 -2.44363341e-02
  1.21598206e-02 -6.59262985e-02  7.89191872e-02  2.19990965e-02
  2.90414616e-02 -3.49428

In [ ]:
for doc in collection.find({'plot':{"$exists":True}}).limit(500):
  embedding = model.encode(doc['plot'])
  doc['plot_embedding_hf']=embedding.tolist()
  collection.replace_one({'_id':doc['_id']},doc)

In [ ]:
query = "good person turns evil"


results = collection.aggregate([
  {"$vectorSearch": {
    "queryVector": model.encode(query).tolist(),
    "path": "plot_embedding_hf",
    "numCandidates": 100,
    "limit": 4,
    "index": "PlotSemanticSearch",
      }}
]);


In [ ]:
for movie in results:
    print(f"Title: {movie.get('title')}")
    print(f"Plot: {movie.get('plot')}")
    print(f"Year: {movie.get('year')}\n")

Title: Dr. Jekyll and Mr. Hyde
Plot: Dr. Jekyll faces horrible consequences when he lets his dark side run wild with a potion that changes him into the animalistic Mr. Hyde.
Year: 1931

Title: Scarface
Plot: An ambitious and near insanely violent gangster climbs the ladder of success in the mob, but his weaknesses prove to be his downfall.
Year: 1932

Title: The Sound of Fury
Plot: A man who is down on his luck falls in with a criminal.
Year: 1950

Title: The Lower Depths
Plot: A charismatic thief makes friends with a bankrupt baron who comes to live in the thief's slum. Meanwhile the thief seeks the love of a young woman, who is held emotionally captive by her slumlord family.
Year: 1936

